In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import numpy as np

In [2]:
df=pd.read_csv("loan_data.csv")
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
x=df.drop(columns='loan_status')
y=df.loan_status

In [4]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,random_state=42,train_size=0.8)

In [6]:
x.corr(numeric_only=True)

num_cols = x.select_dtypes(include='number').columns
obj_cols = x.select_dtypes(include='object').columns


C:\Users\lahar\AppData\Local\Temp\ipykernel_28356\494637197.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = x.select_dtypes(include='object').columns


In [7]:
x[obj_cols].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [9]:
onehot_col=['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']
ordinal_col=['person_education']
order=['High School','Bachelor','Master','Associate','Doctorate']
print(df["person_education"].unique())
model=KNeighborsClassifier()
preprocessing=ColumnTransformer(
    transformers=[
        ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False),onehot_col),
        ('ordinal',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),ordinal_col)
    ],
    remainder="passthrough"
)

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str


In [10]:
model=DecisionTreeClassifier(random_state=42)

In [11]:
main_pipeline=Pipeline([
    ('preprocessing',preprocessing),
    ('model',DecisionTreeClassifier(random_state=42))
])

In [12]:
grid_search_cv=GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion':['gini','entropy'],
        'model__max_depth':[None,5,10,50,100],
        'model__min_samples_split':[2,5,7,10],
        'model__min_samples_leaf':[1,3,5,7,10],
        'model__splitter':['best','random']
    },
    verbose=1
)
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 400 candidates, totalling 2000 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 5, ...], 'model__min_samples_leaf': [1, 3, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a

In [13]:
grid_search_cv.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['person_age','person_gender','person_education',..., 'cb_person_cred_hist_length','credit_score', 'previous_loan_defaults_on_file']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

In [14]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 10,
 'model__min_samples_leaf': 1,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [15]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.15043511, 0.07943692, 0.1258019 , 0.07744889, 0.12636557,
        0.0756566 , 0.12720499, 0.07485542, 0.12398381, 0.07584472,
        0.12216072, 0.07664785, 0.12658706, 0.07875633, 0.12938118,
        0.07800264, 0.12328143, 0.079878  , 0.13028278, 0.08633652,
        0.13703632, 0.08061471, 0.132938  , 0.08122764, 0.13221002,
        0.07889886, 0.13226976, 0.07818031, 0.12719078, 0.07801728,
        0.13828001, 0.07269664, 0.13586097, 0.0825707 , 0.13598261,
        0.07326794, 0.12990546, 0.07132502, 0.11611557, 0.07050633,
        0.08147545, 0.06320167, 0.07829437, 0.05893402, 0.07988205,
        0.05957828, 0.08178329, 0.06161976, 0.07950292, 0.05912361,
        0.08134789, 0.0610558 , 0.0826611 , 0.05991578, 0.0806757 ,
        0.05950284, 0.0811317 , 0.05970836, 0.07930956, 0.06362205,
        0.08356552, 0.06357336, 0.09829745, 0.073596  , 0.08181305,
        0.0687449 , 0.08488989, 0.0703577 , 0.09045286, 0.07011218,
        0.08520198, 0.0770587 ,

In [16]:
results=pd.DataFrame(grid_search_cv.cv_results_)

In [17]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
280,0.151919,0.022209,0.017102,0.003037,entropy,10,1,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.918889,0.925972,0.914861,0.921111,0.920833,0.920333,0.003597,1
302,0.142072,0.006168,0.018155,0.003356,entropy,10,5,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
298,0.224814,0.029460,0.024036,0.001976,entropy,10,5,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
300,0.174043,0.026748,0.023526,0.005410,entropy,10,5,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.917917,0.927361,0.914444,0.921250,0.920417,0.920278,0.004260,2
282,0.128112,0.007712,0.014281,0.000583,entropy,10,1,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.919028,0.925972,0.915000,0.920694,0.920694,0.920278,0.003527,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,0.068745,0.008269,0.014433,0.003360,gini,5,7,2,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872639,0.872417,0.000461,393
79,0.059184,0.001060,0.012018,0.000379,gini,5,10,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
77,0.064795,0.005545,0.014613,0.004622,gini,5,10,7,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
75,0.077446,0.014076,0.013936,0.002441,gini,5,10,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397


In [18]:
ypred_test=grid_search_cv.predict(xtest)

In [19]:
print(classification_report(ytest,ypred_test))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      6990
           1       0.89      0.73      0.80      2010

    accuracy                           0.92      9000
   macro avg       0.91      0.85      0.88      9000
weighted avg       0.92      0.92      0.92      9000

